In [ ]:
# OPTIONAL: Kiem tra nhanh runtime device.
# Notebook 01 co the chay CPU; GPU khong bat buoc cho orchestration.
try:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Dang su dung thiet bi: {device}")
except ImportError:
    print("torch chua duoc cai trong runtime nay. Bo qua kiem tra device; Notebook 01 khong bat buoc GPU.")


## Harness smoke readiness markers

GitHub repo setup uses `git clone` on first run and `git fetch/checkout` on later runs. It keeps the same operator markers as Notebook 00, including `git pull` compatibility wording, then installs System 1 with `pip install -e`.

Runtime roots used by local, Colab, and Kaggle execution: `AIC_REPO_PARENT`, `AIC_REPO_ROOT`, `AIC_WORKSPACE`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, `AIC_ARTIFACT_ROOT`, `AIC_OUTPUT_DIR`, `AIC_RELEASE_ID`, `AIC_BATCH_ID`, `AIC_WORKER_ID`, and `AIC_HF_REPO_ID`.

Notebook 01 phase01 handoff uses `restore-phase00-ingestion`, `process-batch`, and `sync-structure-artifacts`.


# Notebook 01 — Worker Structure Pipeline

Notebook này là orchestration mỏng cho **phase01_structure** của System 1:

```text
AIC26_release/{release_id}/phase00_ingestion
→ restore-phase00-ingestion
→ materialize tables/raw_mapping/frame_timeline/manifests vào local active release root
→ process-batch timeline-aware cho một batch
→ tạo structure ZIP + worker report
→ sync-structure-artifacts lên AIC26_release/{release_id}/phase01_structure
→ preview/check output
```

Thiết kế giống Notebook 00:

```text
Notebook = config + environment setup + gọi CLI + verify/report
Package/CLI = business logic thật
```

Notebook này **không** làm feature extraction, phase02, merge, build-index, build-db, validate final release, hoặc release packaging. Các bước đó thuộc Notebook 02/03.

Điểm quan trọng:

- `release_id` phải trùng Notebook 00.
- `batch_id` phải tồn tại trong `phase00_ingestion/manifests/`.
- `worker_id` nên duy nhất theo runtime/máy.
- Không truyền `canonical_release_vXXX/phase00_ingestion` vào `hf_prefix`; command tự thêm phase path.
- Notebook chỉ gọi CLI. Không tự copy file phase00; package `restore-phase00-ingestion` đã materialize active layout cho `process-batch`.
- Production mặc định `require_frame_timeline=True`: mọi video trong batch phải có timeline status `pass` và file `frame_timeline/{video_id}.parquet`.
- `process-batch --require-frame-timeline` dừng ngay nếu timeline thiếu/hỏng; notebook không tự tính frame bằng FPS trong cell.


In [ ]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass

@dataclass
class WorkflowConfig:
    # 1. Hugging Face processed/release repo.
    hf_release_repo: str = os.environ.get("AIC_HF_REPO_ID", "1thesudden/AIC26_release")
    hf_repo_type: str = os.environ.get("AIC_HF_REPO_TYPE", "dataset")
    hf_revision: str = os.environ.get("AIC_HF_REVISION", "main")
    hf_prefix: str = os.environ.get("AIC_HF_PREFIX", "")

    # 2. Contract ids. Must match Notebook 00 output.
    release_id: str = os.environ.get("AIC_RELEASE_ID", "canonical_release_v009")
    batch_id: str = os.environ.get("AIC_BATCH_ID", "batch_000")
    worker_id: str = os.environ.get("AIC_WORKER_ID", "worker_000")

    # 3. Repo code.
    github_repo_url: str = os.environ.get("AIC_GITHUB_REPO_URL", "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git")
    github_branch: str = os.environ.get("AIC_GITHUB_BRANCH", "system1-notebook01")
    repo_dir_name: str = os.environ.get("AIC_REPO_DIR_NAME", "Multimodal-Agentic-Retrieval-Engine")
    use_existing_repo: bool = os.environ.get("AIC_USE_EXISTING_REPO", "1") != "0"

    # 4. Runtime paths.
    workspace: Path = Path(os.environ.get("AIC_WORKSPACE", "/content/system1_workspace" if "google.colab" in sys.modules else Path.cwd())).expanduser().resolve()
    output_dir: str = os.environ.get("AIC_OUTPUT_DIR", str(workspace / "output"))
    input_dir: str = os.environ.get("AIC_INPUT_DIR", str(workspace / "input"))
    data_root: str = os.environ.get("AIC_DATA_ROOT", str(workspace / "data"))
    runtime_root: str = os.environ.get("AIC_RUNTIME_ROOT", str(workspace / "runtime"))
    artifact_root: str = os.environ.get("AIC_ARTIFACT_ROOT", output_dir)

    # 5. Workflow options.
    providers: str = os.environ.get("AIC_PROVIDERS", "mock")
    require_frame_timeline: bool = os.environ.get("AIC_REQUIRE_FRAME_TIMELINE", "1") != "0"

    # 6. Workflow switches.
    run_restore_phase00: bool = os.environ.get("AIC_RUN_RESTORE_PHASE00", "1") != "0"
    run_process_batch: bool = os.environ.get("AIC_RUN_PROCESS_BATCH", "1") != "0"
    run_sync_structure: bool = os.environ.get("AIC_RUN_SYNC_STRUCTURE", "1") != "0"
    run_check_hf_structure: bool = os.environ.get("AIC_RUN_CHECK_HF_STRUCTURE", "1") != "0"
    run_restore_verify: bool = os.environ.get("AIC_RUN_RESTORE_VERIFY", "0") == "1"

    # process-batch normally does not need --input when media_store_manifest has canonical HF columns.
    use_local_input_dir: bool = os.environ.get("AIC_USE_LOCAL_INPUT_DIR", "0") == "1"

    # Keep Notebook 01 explicit: no hidden checkpoint/sync path inside process-batch.
    process_resume: bool = os.environ.get("AIC_PROCESS_RESUME", "0") == "1"
    process_implicit_sync: bool = os.environ.get("AIC_PROCESS_IMPLICIT_SYNC", "0") == "1"
    restore_overwrite: bool = os.environ.get("AIC_RESTORE_OVERWRITE", "1") != "0"

    def __post_init__(self):
        if "google.colab" in sys.modules:
            self.env = "colab"
        elif "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
            self.env = "kaggle"
        else:
            self.env = "local"

        self.workspace.mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).expanduser().mkdir(parents=True, exist_ok=True)
        Path(self.input_dir).expanduser().mkdir(parents=True, exist_ok=True)
        Path(self.data_root).expanduser().mkdir(parents=True, exist_ok=True)
        Path(self.runtime_root).expanduser().mkdir(parents=True, exist_ok=True)
        Path(self.artifact_root).expanduser().mkdir(parents=True, exist_ok=True)

        os.environ["AIC_RELEASE_ID"] = self.release_id
        os.environ["AIC_HF_REPO_ID"] = self.hf_release_repo
        os.environ["AIC_HF_REPO_TYPE"] = self.hf_repo_type
        os.environ["AIC_HF_REVISION"] = self.hf_revision
        os.environ["AIC_HF_PREFIX"] = self.hf_prefix
        os.environ["AIC_OUTPUT_ROOT"] = self.output_dir
        os.environ["AIC_DATA_ROOT"] = self.data_root
        os.environ["AIC_RUNTIME_ROOT"] = self.runtime_root
        os.environ["AIC_ARTIFACT_ROOT"] = self.artifact_root
        os.environ["AIC_SYNC"] = "false"
        os.environ["AIC_RESUME"] = "false"

config = WorkflowConfig()

print("Moi truong:", config.env)
print("Workspace:", config.workspace)
print("Output dir:", config.output_dir)
print("Input dir:", config.input_dir)
print("Data root:", config.data_root)
print("Runtime root:", config.runtime_root)
print("Artifact root:", config.artifact_root)
print("Release ID:", config.release_id)
print("Batch ID:", config.batch_id)
print("Worker ID:", config.worker_id)
print("HF release repo:", config.hf_release_repo)
print("HF prefix:", repr(config.hf_prefix))
print("Providers:", config.providers)
print("Require frame timeline:", config.require_frame_timeline)
print("Run restore phase00:", config.run_restore_phase00)
print("Run process batch:", config.run_process_batch)
print("Run sync structure:", config.run_sync_structure)
print("Use local input dir:", config.use_local_input_dir)


In [ ]:
# BUOC 1: Configure environment va HF token.
import os
import sys
import subprocess
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ.setdefault("AIC_VERBOSE", "0")
os.environ.setdefault("AIC_HF_PROGRESS", "0")

if config.env == "colab":
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["AIC_HF_TOKEN"] = hf_token
    except Exception as exc:
        print("Khong doc duoc Colab Secret HF_TOKEN:", exc)
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["AIC_HF_TOKEN"] = hf_token

if os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN"):
    print("HF token configured.")
else:
    print("Canh bao: chua thay HF_TOKEN/AIC_HF_TOKEN. Restore/sync HF se fail neu repo can token.")

print("Notebook 01 environment ready.")


In [ ]:
# BUOC 2: Resolve repo code local hoac clone/sync repo.
from pathlib import Path
import os
import subprocess


def looks_like_repo_root(path: Path) -> bool:
    return (path / "system1" / "src" / "system1" / "__init__.py").exists() or (path / "src" / "system1" / "__init__.py").exists()


def find_existing_repo(start: Path) -> Path | None:
    start = start.expanduser().resolve()
    for candidate in [start, *start.parents]:
        if looks_like_repo_root(candidate):
            return candidate
    return None


def run_git(cmd, *, cwd=None, check=True):
    safe_cwd = Path(cwd).expanduser().resolve() if cwd else config.workspace
    safe_cwd.mkdir(parents=True, exist_ok=True)

    print("CWD:", safe_cwd)
    print("RUN:", " ".join(map(str, cmd)))

    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(safe_cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if result.stdout:
        print(result.stdout)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit_code={result.returncode}\n"
            f"CMD: {' '.join(map(str, cmd))}\n\n"
            f"OUTPUT:\n{result.stdout}"
        )

    return result


existing_repo = find_existing_repo(Path.cwd()) if config.use_existing_repo else None
repo_dir = existing_repo or (config.workspace / config.repo_dir_name).expanduser().resolve()
os.environ["AIC_REPO_ROOT"] = str(repo_dir)
os.environ["AIC_REPO_PARENT"] = str(repo_dir.parent)

print("Current cwd:", Path.cwd())
print("workspace:", config.workspace)
print("repo_dir:", repo_dir)
print("use_existing_repo:", config.use_existing_repo)
print("repo_dir exists:", repo_dir.exists())
print("repo_dir .git exists:", (repo_dir / ".git").exists())
print("github_repo_url:", config.github_repo_url)
print("github_branch:", config.github_branch)

if (repo_dir / ".git").exists():
    print("Dung/cap nhat repo an toan:", repo_dir)
    dirty = run_git(["git", "status", "--porcelain"], cwd=repo_dir, check=False)
    if dirty.stdout.strip():
        raise RuntimeError("Repo co thay doi local. Hay commit/stash thu cong; notebook khong reset hoac xoa thay doi.")
    run_git(["git", "fetch", "origin", "--prune"], cwd=repo_dir)
    remote_branch = f"origin/{config.github_branch}"
    branch_check = run_git(["git", "rev-parse", "--verify", remote_branch], cwd=repo_dir, check=False)
    if branch_check.returncode != 0:
        raise RuntimeError(f"Khong tim thay remote branch {remote_branch}.")
    local_branch = run_git(["git", "rev-parse", "--verify", f"refs/heads/{config.github_branch}"], cwd=repo_dir, check=False)
    if local_branch.returncode == 0:
        run_git(["git", "switch", config.github_branch], cwd=repo_dir)
    else:
        run_git(["git", "switch", "--track", "-c", config.github_branch, remote_branch], cwd=repo_dir)
    run_git(["git", "merge", "--ff-only", remote_branch], cwd=repo_dir)
else:
    if repo_dir.exists():
        raise RuntimeError(
            f"repo_dir ton tai nhung khong phai git repo hoac khong nhan dien duoc source layout: {repo_dir}. "
            "Hay set AIC_WORKSPACE/AIC_REPO_DIR_NAME khac hoac xoa folder nay thu cong."
        )
    print("Clone repo:", config.github_repo_url)
    run_git(["git", "clone", "--branch", config.github_branch, "--single-branch", config.github_repo_url, str(repo_dir)], cwd=config.workspace)

print("Git commit hien tai:")
run_git(["git", "log", "-1", "--oneline"], cwd=repo_dir, check=False)

print("Git status:")
run_git(["git", "status", "--short"], cwd=repo_dir, check=False)


In [ ]:
# BUOC 2B: Kiem tra project_root va setup import path cho notebook kernel.
import sys
import subprocess
from pathlib import Path

print("\nCheck package source layout:")

project_root_candidates = [
    repo_dir / "system1",
    repo_dir,
]

project_root = None
for candidate in project_root_candidates:
    print("- candidate:", candidate)
    print("  has src/system1:", (candidate / "src" / "system1").exists())
    print("  has src/system1/__init__.py:", (candidate / "src" / "system1" / "__init__.py").exists())
    print("  has runtime/environment.py:", (candidate / "src" / "system1" / "runtime" / "environment.py").exists())

    if (candidate / "src" / "system1" / "__init__.py").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise RuntimeError(
        "Khong tim thay project_root chua src/system1/__init__.py.\n"
        "Checked:\n" + "\n".join(str(p) for p in project_root_candidates)
    )

system1_src = project_root / "src"

print("\nInstalling editable package from:", project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root)],
    check=True,
)

system1_src_str = str(system1_src)
sys.path = [system1_src_str] + [
    p for p in sys.path
    if str(Path(p).expanduser().resolve()) != system1_src_str
]

for module_name in list(sys.modules):
    if module_name == "system1" or module_name.startswith("system1."):
        del sys.modules[module_name]

import system1

REPO_ROOT = repo_dir
SYSTEM1_ROOT = project_root

print("\nResolved project paths:")
print("- REPO_ROOT:", REPO_ROOT)
print("- SYSTEM1_ROOT:", SYSTEM1_ROOT)
print("- system1_src:", system1_src)
print("- system1 file:", getattr(system1, "__file__", None))
print("- system1 path:", list(getattr(system1, "__path__", [])))

if getattr(system1, "__file__", None) is None:
    raise RuntimeError(
        "system1 van dang bi import thanh namespace package. "
        "Hay restart runtime roi chay lai tu BUOC 1."
    )

system1_file = Path(system1.__file__).resolve()
expected_package_dir = (system1_src / "system1").resolve()
if system1_file.parent != expected_package_dir:
    raise RuntimeError(
        "Kernel dang import system1 tu source khac repo vua sync: "
        f"actual={system1_file}, expected_under={expected_package_dir}. "
        "Hay restart runtime roi chay lai tu BUOC 1."
    )
print("- package source preflight: OK")


In [ ]:
# BUOC 3: Dinh nghia helper run_cli de goi system1 CLI trong notebook.
def run_cli(args, *, check=True, stream=True, tail_lines=300):
    import os
    import subprocess
    import sys
    from pathlib import Path

    def find_repo_root():
        candidates = [
            globals().get("SYSTEM1_ROOT"),
            globals().get("REPO_ROOT"),
            Path.cwd(),
            Path(config.workspace) / config.repo_dir_name,
            Path(config.workspace),
        ]

        for candidate in candidates:
            if candidate is None:
                continue
            path = Path(candidate).expanduser().resolve()

            if path.name == "system1" and (path / "src" / "system1").exists():
                return path

            if (path / "system1" / "src" / "system1").exists():
                return path

        raise RuntimeError(
            "Khong tim thay repo root. Hay chay cell clone/setup repo truoc, "
            "hoac kiem tra AIC_WORKSPACE/AIC_REPO_DIR_NAME co tro dung repo local khong."
        )

    cli_cwd = find_repo_root()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    if (cli_cwd / "src" / "system1").exists():
        system1_src = cli_cwd / "src"
    else:
        system1_src = cli_cwd / "system1" / "src"

    env["PYTHONPATH"] = str(system1_src) + os.pathsep + env.get("PYTHONPATH", "")

    if env.get("AIC_HF_PROGRESS", "0") != "1":
        env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        env["HF_HUB_VERBOSITY"] = "error"

    env.setdefault("AIC_VERBOSE", "0")
    env["AIC_RELEASE_ID"] = config.release_id
    env["AIC_HF_REPO_ID"] = config.hf_release_repo
    env["AIC_HF_REPO_TYPE"] = config.hf_repo_type
    env["AIC_HF_REVISION"] = config.hf_revision
    env["AIC_HF_PREFIX"] = config.hf_prefix
    env["AIC_OUTPUT_ROOT"] = config.output_dir

    cmd = [sys.executable, "-m", "system1.cli", *args]

    print("\n" + "=" * 100)
    print("CWD:", cli_cwd)
    print("PYTHONPATH prefix:", system1_src)
    print("RUN:", " ".join(cmd))
    print("stream:", stream)
    print("=" * 100)

    if not stream:
        completed = subprocess.run(
            cmd,
            cwd=str(cli_cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

        output = completed.stdout or ""
        output_lines = output.splitlines()
        tail = "\n".join(output_lines[-tail_lines:])

        print(f"CLI finished: exit_code={completed.returncode}")
        print(f"Last {min(tail_lines, len(output_lines))} lines:")
        print("-" * 100)
        print(tail)
        print("-" * 100)

        if check and completed.returncode != 0:
            raise RuntimeError(
                f"CLI failed with exit code {completed.returncode}: {' '.join(args)}\n\n"
                f"Last output:\n{tail}"
            )

        return output

    process = subprocess.Popen(
        cmd,
        cwd=str(cli_cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    returncode = process.wait()
    output = "".join(lines)

    if check and returncode != 0:
        tail = "\n".join(output.splitlines()[-tail_lines:])
        raise RuntimeError(
            f"CLI failed with exit code {returncode}: {' '.join(args)}\n\n"
            f"Last output:\n{tail}"
        )

    return output


In [ ]:
# BUOC 4: Resolve runtime paths.
from pathlib import Path
from system1.runtime.environment import resolve_runtime_paths

runtime_paths = resolve_runtime_paths(output_root=Path(config.output_dir), release_id=config.release_id)
output_base = runtime_paths.output_root
input_base = Path(config.input_dir).expanduser().resolve()
release_root = output_base / config.release_id

output_base.mkdir(parents=True, exist_ok=True)
input_base.mkdir(parents=True, exist_ok=True)

print("Runtime paths:")
print("- output_base:", output_base)
print("- input_base:", input_base)
print("- release_root:", release_root)
print("- release_id:", config.release_id)
print("- batch_id:", config.batch_id)
print("- worker_id:", config.worker_id)

# Không chạy preflight bằng output `--help`: giao diện Rich có thể thay đổi theo terminal.
# Các command thật bên dưới chạy từ source repo đã sync và tự validate đầy đủ option/input.


In [ ]:
# BUOC 5: Restore phase00_ingestion tu AIC26_release.
# Command giu snapshot o output/{release_id}/phase00_ingestion va materialize active layout cho process-batch.
from pathlib import Path

print("BUOC 5: Restore phase00_ingestion")
print("- hf_release_repo:", config.hf_release_repo)
print("- release_id:", config.release_id)
print("- output_base:", output_base)
print("- restore_overwrite:", config.restore_overwrite)

if config.run_restore_phase00:
    restore_cmd = [
        "restore-phase00-ingestion",
        "--release-id", config.release_id,
        "--hf-repo-id", config.hf_release_repo,
        "--hf-prefix", config.hf_prefix,
        "--hf-repo-type", config.hf_repo_type,
        "--hf-revision", config.hf_revision,
        "--output", str(output_base),
    ]
    restore_cmd.append("--overwrite" if config.restore_overwrite else "--no-overwrite")
    run_cli(restore_cmd, stream=False, tail_lines=300)
else:
    print("Bo qua restore-phase00-ingestion theo config.")

release_root = output_base / config.release_id
phase00_root = release_root / "phase00_ingestion"
print("release_root:", release_root)
print("phase00_root exists:", phase00_root.exists())
print("active tables exists:", (release_root / "tables").exists())
print("active raw_mapping exists:", (release_root / "raw_mapping").exists())
print("active frame_timeline exists:", (release_root / "frame_timeline").exists())
print("active manifests exists:", (release_root / "manifests").exists())


In [ ]:
# BUOC 6: Kiem tra input local toi thieu cho process-batch.
from pathlib import Path
import pandas as pd

release_root = output_base / config.release_id
required_paths = [
    release_root / "phase00_ingestion",
    release_root / "tables" / "videos.parquet",
    release_root / "raw_mapping" / "media_store_manifest.parquet",
    release_root / "manifests" / "frame_timeline_manifest.parquet",
    release_root / "manifests" / f"{config.batch_id}.txt",
]
missing = [p for p in required_paths if not p.exists()]
if missing:
    raise RuntimeError("Thieu input cho process-batch:\n" + "\n".join(str(p) for p in missing))

batch_path = release_root / "manifests" / f"{config.batch_id}.txt"
batch_video_ids = [line.strip() for line in batch_path.read_text(encoding="utf-8").splitlines() if line.strip()]
if not batch_video_ids:
    raise RuntimeError(f"Batch file rong: {batch_path}")

videos_df = pd.read_parquet(release_root / "tables" / "videos.parquet")
media_df = pd.read_parquet(release_root / "raw_mapping" / "media_store_manifest.parquet")
frame_timeline_manifest_df = pd.read_parquet(release_root / "manifests" / "frame_timeline_manifest.parquet")

missing_video_rows = sorted(set(batch_video_ids) - set(videos_df["video_id"].astype(str)))
missing_mapping_rows = sorted(set(batch_video_ids) - set(media_df["video_id"].astype(str)))

print(f"Batch {config.batch_id}: {len(batch_video_ids)} videos")
print("videos.parquet rows:", len(videos_df))
print("media_store_manifest rows:", len(media_df))
timeline_batch_df = frame_timeline_manifest_df[frame_timeline_manifest_df["video_id"].astype(str).isin(batch_video_ids)] if "video_id" in frame_timeline_manifest_df.columns else frame_timeline_manifest_df.iloc[0:0]
timeline_status_counts = timeline_batch_df["status"].value_counts(dropna=False).to_dict() if "status" in timeline_batch_df.columns else {}
print("frame_timeline_manifest rows:", len(frame_timeline_manifest_df))
print("batch frame_timeline rows:", len(timeline_batch_df))
print("batch frame_timeline status counts:", timeline_status_counts)
print("missing video rows:", missing_video_rows[:20])
print("missing mapping rows:", missing_mapping_rows[:20])

if missing_video_rows:
    raise RuntimeError(f"Batch references videos missing from videos.parquet: {missing_video_rows[:20]}")
if missing_mapping_rows:
    raise RuntimeError(f"Batch references videos missing from media_store_manifest.parquet: {missing_mapping_rows[:20]}")
missing_timeline_rows = sorted(set(batch_video_ids) - set(timeline_batch_df["video_id"].astype(str))) if "video_id" in timeline_batch_df.columns else batch_video_ids
print("missing frame_timeline manifest rows:", missing_timeline_rows[:20])
if missing_timeline_rows:
    raise RuntimeError(f"Batch references videos missing from frame_timeline_manifest.parquet: {missing_timeline_rows[:20]}")
invalid_timeline_ids = sorted(set(batch_video_ids) - set(timeline_batch_df.loc[timeline_batch_df["status"].astype(str) == "pass", "video_id"].astype(str)))
missing_timeline_files = [release_root / "frame_timeline" / f"{video_id}.parquet" for video_id in batch_video_ids if not (release_root / "frame_timeline" / f"{video_id}.parquet").exists()]
if config.require_frame_timeline and (invalid_timeline_ids or missing_timeline_files):
    raise RuntimeError(f"Production batch thiếu timeline pass/file: invalid_ids={invalid_timeline_ids[:20]}, missing_files={missing_timeline_files[:20]}")

canonical_cols = [c for c in ["canonical_backend", "canonical_repo_id", "canonical_prefix", "canonical_video_path", "canonical_metadata_path"] if c in media_df.columns]
print("canonical columns present:", canonical_cols)
print("First batch ids:")
for video_id in batch_video_ids[:20]:
    print("-", video_id)
if len(batch_video_ids) > 20:
    print(f"... {len(batch_video_ids) - 20} more")

print("process-batch input OK.")


In [ ]:
# BUOC 7: Run timeline-aware process-batch de tao structure artifacts local.
from pathlib import Path

print("BUOC 7: process-batch")
print("- providers:", config.providers)
print("- batch_id:", config.batch_id)
print("- worker_id:", config.worker_id)
print("- use_local_input_dir:", config.use_local_input_dir)
print("- frame_timeline_manifest:", release_root / "manifests" / "frame_timeline_manifest.parquet")

if config.run_process_batch:
    process_cmd = [
        "process-batch",
        "--worker-id", config.worker_id,
        "--batch-id", config.batch_id,
        "--providers", config.providers,
        "--output", str(output_base),
    ]
    process_cmd.append("--require-frame-timeline" if config.require_frame_timeline else "--allow-missing-frame-timeline")
    if config.use_local_input_dir:
        process_cmd.extend(["--input", str(input_base)])
    process_cmd.append("--resume" if config.process_resume else "--no-resume")
    process_cmd.append("--sync" if config.process_implicit_sync else "--no-sync")
    run_cli(process_cmd, stream=True, tail_lines=300)
else:
    print("Bo qua process-batch theo config.")

structure_dir = release_root / "artifacts" / "structure"
worker_reports_dir = release_root / "manifests" / "worker_reports"

print("structure_dir:", structure_dir)
print("worker_reports_dir:", worker_reports_dir)
print("structure zip count:", len(list(structure_dir.glob("*_structure.zip"))) if structure_dir.exists() else 0)
print("worker reports:", sorted(p.name for p in worker_reports_dir.glob("structure_*.json")) if worker_reports_dir.exists() else [])


In [ ]:
# BUOC 8: Kiem tra structure ZIP + worker report truoc khi sync.
from pathlib import Path
import json

structure_dir = release_root / "artifacts" / "structure"
expected_zips = [structure_dir / f"{video_id}_structure.zip" for video_id in batch_video_ids]
missing_zips = [p for p in expected_zips if not p.exists()]
if missing_zips:
    raise RuntimeError("Thieu structure ZIP:\n" + "\n".join(str(p) for p in missing_zips[:50]))

report_path = release_root / "manifests" / "worker_reports" / f"structure_{config.batch_id}_{config.worker_id}.json"
if not report_path.exists():
    raise RuntimeError(f"Thieu worker report: {report_path}")

print(f"Structure ZIP OK: {len(expected_zips)} files")
print("Worker report:", report_path)

report = json.loads(report_path.read_text(encoding="utf-8"))
for key in ["phase", "batch_id", "worker_id", "started_at", "finished_at", "videos_processed", "videos_failed", "status", "legacy_status", "video_count", "error_count"]:
    if key in report:
        print(f"{key}: {report[key]}")

if report.get("batch_id") != config.batch_id:
    raise RuntimeError(f"Worker report batch_id mismatch: {report.get('batch_id')} != {config.batch_id}")
if report.get("worker_id") != config.worker_id:
    raise RuntimeError(f"Worker report worker_id mismatch: {report.get('worker_id')} != {config.worker_id}")


In [ ]:
# BUOC 9: Sync structure artifacts len AIC26_release/phase01_structure.
print("BUOC 9: sync-structure-artifacts")
print("- hf_release_repo:", config.hf_release_repo)
print("- release_id:", config.release_id)
print("- batch_id:", config.batch_id)
print("- worker_id:", config.worker_id)

if config.run_sync_structure:
    sync_cmd = [
        "sync-structure-artifacts",
        "--release-id", config.release_id,
        "--batch-id", config.batch_id,
        "--worker-id", config.worker_id,
        "--hf-repo-id", config.hf_release_repo,
        "--hf-prefix", config.hf_prefix,
        "--hf-repo-type", config.hf_repo_type,
        "--hf-revision", config.hf_revision,
        "--output", str(output_base),
    ]
    run_cli(sync_cmd, stream=False, tail_lines=300)
else:
    print("Bo qua sync-structure-artifacts theo config.")


In [ ]:
# BUOC 10: Kiem tra HF phase01_structure layout sau sync.
from pathlib import Path
import os

try:
    from huggingface_hub import HfApi
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi

repo_id = config.hf_release_repo
release_id = config.release_id
phase01_root = f"{release_id}/phase01_structure"
artifact_prefix = f"{phase01_root}/artifacts/{config.batch_id}/"
report_prefix = f"{phase01_root}/worker_reports/"

if not config.run_check_hf_structure:
    print("Bo qua HF phase01_structure check theo config.")
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Thieu HF_TOKEN/AIC_HF_TOKEN de kiem tra HF repo.")

    api = HfApi(token=hf_token)
    files = sorted(entry.path for entry in api.list_repo_tree(repo_id=repo_id, repo_type=config.hf_repo_type, revision=config.hf_revision, path_in_repo=phase01_root, recursive=True, token=hf_token) if getattr(entry, "path", None))
    phase01_files = [p for p in files if p.startswith(f"{phase01_root}/")]
    artifact_files = [p for p in phase01_files if p.startswith(artifact_prefix) and p.endswith("_structure.zip")]
    report_files = [p for p in phase01_files if p.startswith(report_prefix) and Path(p).name == f"structure_{config.batch_id}_{config.worker_id}.json"]

    print("HF release repo:", repo_id)
    print("phase01_root:", phase01_root)
    print("phase01_file_count:", len(phase01_files))
    print("artifact_files count:", len(artifact_files))
    print("worker report files:", report_files)

    missing_remote_zips = [
        f"{artifact_prefix}{video_id}_structure.zip"
        for video_id in batch_video_ids
        if f"{artifact_prefix}{video_id}_structure.zip" not in files
    ]

    print("missing_remote_zips:", missing_remote_zips[:20])
    if missing_remote_zips:
        raise RuntimeError("HF phase01_structure thieu structure ZIP:\n" + "\n".join(missing_remote_zips[:50]))
    if not report_files:
        raise RuntimeError("HF phase01_structure thieu worker report cho batch/worker hien tai.")

    print("HF phase01_structure structure OK.")


In [ ]:
# BUOC 11 OPTIONAL: Restore verify phase01_structure vao output khac.
# Buoc nay chi check path restore, khong extract ZIP.
from pathlib import Path

if config.run_restore_verify:
    verify_output = config.workspace / "output_restore_verify"
    run_cli([
        "restore-structure-artifacts",
        "--release-id", config.release_id,
        "--batch-id", config.batch_id,
        "--hf-repo-id", config.hf_release_repo,
        "--hf-prefix", config.hf_prefix,
        "--hf-repo-type", config.hf_repo_type,
        "--hf-revision", config.hf_revision,
        "--output", str(verify_output),
    ], stream=False, tail_lines=300)

    verify_structure_dir = verify_output / config.release_id / "artifacts" / "structure"
    restored_zips = sorted(verify_structure_dir.glob("*_structure.zip"))
    print("verify_structure_dir:", verify_structure_dir)
    print("restored_zips:", [p.name for p in restored_zips[:30]])
    if len(restored_zips) == 0:
        raise RuntimeError("Restore verify khong tai duoc structure ZIP nao.")
else:
    print("Bo qua restore verify theo config.")


In [ ]:
# BUOC 12: Preview output Notebook 01.
import json
from pathlib import Path

summary = {
    "release_id": config.release_id,
    "batch_id": config.batch_id,
    "worker_id": config.worker_id,
    "providers": config.providers,
    "video_count": len(batch_video_ids),
    "local_release_root": str(release_root),
    "local_structure_dir": str(release_root / "artifacts" / "structure"),
    "local_worker_report": str(release_root / "manifests" / "worker_reports" / f"structure_{config.batch_id}_{config.worker_id}.json"),
    "frame_timeline_manifest": str(release_root / "manifests" / "frame_timeline_manifest.parquet"),
    "batch_frame_timeline_status_counts": timeline_status_counts,
    "hf_structure_prefix": f"{config.release_id}/phase01_structure/artifacts/{config.batch_id}/",
    "hf_worker_report_prefix": f"{config.release_id}/phase01_structure/worker_reports/",
}

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nNotebook 01 hoan tat neu cell nay chay xong.")
